In [27]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
from sklearn.metrics import confusion_matrix



In [2]:
# Inpsect the dataset

data = np.load("processed_dataset.npz", allow_pickle=True)

print(data.files)

for key in data.files:
    print(key, data[key].shape)

features_columns = ['Accel_X', 'Accel_Y', 'Accel_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z']
label_columns = ['Label']

df1 = pd.DataFrame(data['x_train'][0], columns=features_columns)

print(data['y_train'][0])
df1



['x_train', 'y_train', 'x_test', 'y_test']
x_train (15640, 40, 6)
y_train (15640,)
x_test (3911, 40, 6)
y_test (3911,)
T


,Accel_X,Accel_Y,Accel_Z,Gyro_X,Gyro_Y,Gyro_Z
0,5.336676,-3.505111,10.766725,-0.083536,0.521467,0.078873
1,3.045424,-2.487576,9.713276,-0.19172,-0.181994,0.040769
2,4.357447,-2.8491,7.450756,-0.135096,-0.677348,-0.070346
3,3.608062,-3.315969,8.207323,-0.024914,0.125904,-0.112181
4,3.89776,-3.10528,8.418014,0.083936,-0.432069,-0.135363
5,5.549759,-3.205836,9.995792,0.045698,0.084469,-0.220231
6,3.861847,-3.366248,9.057265,-0.066216,-0.018652,-0.10392
7,4.168305,-3.229778,8.8777,-0.088332,-0.204643,-0.133498
8,4.019865,-3.29921,9.129091,-0.072345,-0.213969,-0.088199
9,4.795586,-3.275268,8.576031,-0.013989,-0.166539,-0.111648


In [3]:
# Data PreProcessing

X_train = data['x_train'].astype(np.float32) # float because neural network performs floating point maths
X_test = data['x_test'].astype(np.float32)

encoder = LabelEncoder() # converts text format to numerical format so algorithm can understand it

y_train = encoder.fit_transform(data['y_train']) # fit means learning the unique classes and transform means learn the mapping and apply to data
y_test = encoder.transform(data['y_test']) # learn the mapping and apply to data

print(encoder.classes_)


['C' 'D' 'J' 'T' 'U' 'Wa' 'Wr']


In [4]:
# Converting numpy arrays to tensors

X_train = torch.tensor(X_train) 
X_test = torch.tensor(X_test)

y_train = torch.tensor(y_train, dtype=torch.long) # long because CrossEntropyLoss expects integer class indices
y_test = torch.tensor(y_test, dtype=torch.long)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

torch.Size([15640, 40, 6])
torch.Size([3911, 40, 6])
torch.Size([15640])
torch.Size([3911])


In [5]:
# Binding input and outputs together & dividing data into batches

train_dataset = TensorDataset(X_train, y_train) # binds inputs and output together

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [6]:
# Defining the model

class HAR_Net(nn.Module):

    
    def __init__(self, input_size, hidden_size, num_classes): # creates the layers

        super().__init__()  # initialization of parent class (nn.Module)

        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    # Neural network expects (no of samples(64), no of features(40*6)) so we need to flatten the x which is now (60,40,6)
    def forward(self, x): # define the path input data takes in the network ie passing data through layers

        x = x.view(x.size(0), -1) # flattening => x(64, 40, 6) to x(64,240)

        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x
    

In [7]:
# creating the model

model = HAR_Net(
    input_size=40*6, 
    hidden_size=128,
    num_classes=len(encoder.classes_)
    )

print(model)

HAR_Net(
  (fc1): Linear(in_features=240, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=7, bias=True)
)


In [ ]:
loss_function = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 0.001
)

num_epochs = 10

for epochs in range(num_epochs):
  
  total_loss = 0

  for X_batch, y_batch in train_loader:

    optimizer.zero_grad() # clear old gradients so model dont accumulate gradients

    output = model(X_batch) # forward pass

    loss = loss_function(output, y_batch) # calculate loss

    loss.backward() # calculate gradients

    optimizer.step() # update weights and biases

    # for inspection

    total_loss += loss.item() #.item() converts tensor containing one number to normal python number

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epochs + 1}/{num_epochs}, "
        f"Loss: {average_loss:.4f}"
    )



In [28]:
# Evaluation of model

model.eval()

with torch.no_grad(): # dont calculate or store the gradients
    
    # Accuracy
    output = model(X_test)
    predicted_labels = torch.argmax(output, dim=1)
    correct = predicted_labels == y_test
    accuracy = correct.float().mean()
    print(f"Accuracy : {accuracy * 100}")

    # Confusion Matrix
    cm = confusion_matrix(
        y_test.numpy(),
        predicted_labels.numpy()
    )

    print(cm)


Accuracy : 90.82076263427734
[[193   0   0   8   0   0   0]
 [  2  32   6   4  18  50   0]
 [  1   4 696   9   3  18   3]
 [  5   0   2 970   0   3  30]
 [  0   4   5   4  27  87   1]
 [  4  11   5   3  11 924   3]
 [  0   0   3  38   2  12 710]]
